## Example 1: boto3 functionality test 1

In [ ]:
import boto3
from botocore.exceptions import NoCredentialsError, ClientError
import json

# Example 1: Basic boto3 client creation and service exploration
print("=== Boto3 Basic Client Creation Example ===")

try:
    # Show boto3 version and basic info
    print(f"Boto3 version: {boto3.__version__}")
    
    # List available AWS services
    session = boto3.Session()
    available_services = session.get_available_services()
    print(f"Total available AWS services: {len(available_services)}")
    print("Sample services:", available_services[:10])
    
    # Create clients for popular services (without credentials)
    # These will work for introspection but not actual API calls
    try:
        s3_client = boto3.client('s3', region_name='us-east-1')
        print(f"S3 client created: {type(s3_client).__name__}")
        print(f"S3 service model: {s3_client._service_model.service_name}")
        
        ec2_client = boto3.client('ec2', region_name='us-east-1')
        print(f"EC2 client created: {type(ec2_client).__name__}")
        print(f"EC2 service model: {ec2_client._service_model.service_name}")
        
    except Exception as client_error:
        print(f"Client creation note: {client_error}")
        raise
    
    # Show available regions for a service
    ec2_regions = session.get_available_regions('ec2')
    print(f"EC2 available in {len(ec2_regions)} regions")
    print("Sample regions:", ec2_regions[:5])
    
    print("Basic boto3 client creation completed successfully!")
    
except Exception as e:
    print(f"Error in basic boto3 functionality: {e}")
    raise AssertionError(
        f"boto3 example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: boto3 functionality test 2

In [ ]:
import boto3
from botocore.exceptions import NoCredentialsError, ClientError
import json

# Example 2: Service operation introspection and API exploration
print("=== Boto3 Service API Exploration Example ===")

try:
    # Create S3 client for API exploration
    s3_client = boto3.client('s3', region_name='us-east-1')
    
    # Explore S3 service operations
    s3_operations = [op for op in dir(s3_client) if not op.startswith('_') and callable(getattr(s3_client, op))]
    s3_api_operations = [op for op in s3_operations if not op in ['close', 'exceptions', 'meta']]
    
    print(f"S3 client has {len(s3_api_operations)} API operations")
    print("Common S3 operations:", s3_api_operations[:10])
    
    # Explore operation documentation
    try:
        # Get operation model for list_buckets
        list_buckets_op = s3_client._service_model.operation_model('ListBuckets')
        print(f"\nListBuckets operation:")
        print(f"  HTTP method: {list_buckets_op.http.get('method', 'Unknown')}")
        print(f"  HTTP URI: {list_buckets_op.http.get('requestUri', 'Unknown')}")
        
        # Get operation model for get_object
        get_object_op = s3_client._service_model.operation_model('GetObject')
        print(f"\nGetObject operation:")
        print(f"  HTTP method: {get_object_op.http.get('method', 'Unknown')}")
        print(f"  Required inputs: {list(get_object_op.input_shape.required_members) if get_object_op.input_shape else []}")
        
    except Exception as op_error:
        print(f"Operation introspection note: {op_error}")
        raise
    
    # Explore DynamoDB API
    dynamodb_client = boto3.client('dynamodb', region_name='us-east-1')
    dynamodb_operations = [op for op in dir(dynamodb_client) if not op.startswith('_') and callable(getattr(dynamodb_client, op))]
    dynamodb_api_operations = [op for op in dynamodb_operations if not op in ['close', 'exceptions', 'meta']]
    
    print(f"\nDynamoDB client has {len(dynamodb_api_operations)} API operations")
    print("DynamoDB operations sample:", dynamodb_api_operations[:8])
    
    print("Service API exploration completed successfully!")
    
except Exception as e:
    print(f"Error in service API exploration: {e}")
    raise AssertionError(
        f"boto3 example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: boto3 functionality test 3

In [ ]:
import boto3
from boto3.dynamodb.conditions import Key, Attr
from botocore.exceptions import NoCredentialsError
import json

# Example 3: Resource vs Client interfaces and high-level abstractions  
print("=== Boto3 Resource vs Client Interfaces Example ===")

try:
    # Compare low-level client with high-level resource
    
    # Low-level client interface
    print("=== Low-Level Client Interface ===")
    s3_client = boto3.client('s3', region_name='us-east-1')
    print(f"S3 Client type: {type(s3_client)}")
    print(f"Client methods are direct API mappings")
    
    # High-level resource interface
    print("\n=== High-Level Resource Interface ===")
    s3_resource = boto3.resource('s3', region_name='us-east-1')
    print(f"S3 Resource type: {type(s3_resource)}")
    print("Resource provides object-oriented interface")
    
    # Demonstrate resource collections
    print(f"S3 resource collections: {list(s3_resource.meta.resource_model.collections)}")
    
    # DynamoDB resource example
    print("\n=== DynamoDB Resource Features ===")
    dynamodb_resource = boto3.resource('dynamodb', region_name='us-east-1')
    print(f"DynamoDB Resource type: {type(dynamodb_resource)}")
    
    # Show condition builder capabilities
    print("\n=== DynamoDB Query Conditions ===")
    # These are condition objects, not actual queries
    key_condition = Key('partition_key').eq('value1')
    filter_condition = Attr('attribute').contains('substring')
    
    print(f"Key condition: {key_condition}")
    print(f"Filter condition: {filter_condition}")
    print("Conditions provide pythonic query building")
    
    # Demonstrate session usage
    print("\n=== Session Management ===")
    session = boto3.Session(region_name='us-west-2')
    print(f"Session region: {session.region_name}")
    print(f"Available profiles: {session.available_profiles}")
    
    # Create clients/resources from session
    session_s3 = session.client('s3')
    print(f"Session-based S3 client region: {session_s3.meta.region_name}")
    
    print("Resource vs Client interfaces demonstration completed!")
    
except Exception as e:
    print(f"Error in resource/client demonstration: {e}")
    raise AssertionError(
        f"boto3 example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: boto3 functionality test 4

In [ ]:
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError
from botocore.stub import Stubber
import json
import time

# Example 4: Configuration, error handling, and advanced features
print("=== Boto3 Configuration and Error Handling Example ===")

try:
    # Custom client configuration
    print("=== Client Configuration Options ===")
    
    # Configure retries and timeouts
    config = Config(
        retries={
            'max_attempts': 3,
            'mode': 'adaptive'
        },
        connect_timeout=10,
        read_timeout=10,
        max_pool_connections=50
    )
    
    s3_client = boto3.client('s3', config=config, region_name='us-east-1')
    print(f"S3 client configured with custom retry and timeout settings")
    print(f"Max retry attempts: 3")
    print(f"Retry mode: adaptive")
    print(f"Connect timeout: 10 seconds")
    print(f"Read timeout: 10 seconds")
    
    # Demonstrate error handling patterns
    print("\n=== Error Handling Patterns ===")
    
    # botocore's Stubber answers the call locally with a canned AccessDenied error, so it never
    # reaches AWS and behaves the same with or without credentials or network access.
    stubber = Stubber(s3_client)
    stubber.add_client_error(
        'list_buckets',
        service_error_code='AccessDenied',
        service_message='Access Denied',
        http_status_code=403,
    )
    with stubber:
        try:
            s3_client.list_buckets()
        except ClientError as e:
            error_code = e.response['Error']['Code']
            print(f"✓ ClientError caught - Error code: {error_code}")
        else:
            raise AssertionError("the stubbed list_buckets call did not raise ClientError")
    stubber.assert_no_pending_responses()
    assert error_code == 'AccessDenied', error_code
    
    # Demonstrate pagination
    print("\n=== Pagination Handling ===")
    ec2_client = boto3.client('ec2', region_name='us-east-1')
    
    # Create paginator (won't actually paginate without credentials)
    try:
        paginator = ec2_client.get_paginator('describe_instances')
        print(f"EC2 describe_instances paginator created: {type(paginator)}")
        print("Paginators handle large result sets automatically")
    except Exception as pag_error:
        print(f"Paginator demo note: {type(pag_error).__name__}")
        raise
    
    # Show waiter functionality
    print("\n=== Waiter Functionality ===")
    try:
        waiter = s3_client.get_waiter('bucket_exists')
        print(f"S3 bucket_exists waiter created: {type(waiter)}")
        print("Waiters poll for resource state changes")
        
        # List available waiters
        available_waiters = s3_client.waiter_names
        print(f"S3 available waiters: {available_waiters[:5]}")
    except Exception as wait_error:
        print(f"Waiter demo note: {type(wait_error).__name__}")
        raise
    
    # Show configuration options
    print("\n=== Advanced Configuration Features ===")
    print("Available configuration options:")
    print("• Retry configuration (max_attempts, mode)")
    print("• Timeout settings (connect_timeout, read_timeout)")
    print("• Connection pooling (max_pool_connections)")
    print("• User agent customization")
    print("• Signature version selection")
    print("• Parameter validation control")
    
    print("Configuration and error handling demonstration completed!")
    
except Exception as e:
    print(f"Error in configuration demonstration: {e}")
    raise AssertionError(
        f"boto3 example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: boto3 functionality test 5

In [ ]:
import boto3
import json
from botocore.exceptions import NoCredentialsError
from botocore.config import Config

# Example 5: Comprehensive Boto3 capabilities and AWS ecosystem overview
print("=== Boto3 Comprehensive Capabilities Overview ===")

try:
    print(f"Boto3 version: {boto3.__version__}")
    
    # Show the full scope of AWS services available
    session = boto3.Session()
    all_services = session.get_available_services()
    
    print(f"\n=== AWS Services Coverage ===")
    print(f"Total AWS services supported: {len(all_services)}")
    
    # Categorize major service areas
    compute_services = [s for s in all_services if any(x in s for x in ['ec2', 'lambda', 'batch', 'ecs', 'eks'])]
    storage_services = [s for s in all_services if any(x in s for x in ['s3', 'dynamodb', 'rds', 'redshift'])]
    ml_services = [s for s in all_services if any(x in s for x in ['sagemaker', 'comprehend', 'rekognition', 'textract'])]
    
    print(f"Compute services: {len(compute_services)} - {compute_services[:5]}")
    print(f"Storage/Database services: {len(storage_services)} - {storage_services[:5]}")
    print(f"Machine Learning services: {len(ml_services)} - {ml_services[:5]}")
    
    # Demonstrate multi-service client creation
    print(f"\n=== Multi-Service Client Creation ===")
    services_to_demo = ['s3', 'ec2', 'lambda', 'dynamodb', 'sns', 'sqs', 'cloudwatch', 'iam']
    clients = {}
    
    for service in services_to_demo:
        # Client creation is offline (no credentials, no network): a failure here is a real regression
        clients[service] = boto3.client(service, region_name='us-east-1')
        assert clients[service].meta.service_model.service_name == service, clients[service].meta.service_model.service_name
        print(f"✓ {service.upper()} client created successfully")

    assert sorted(clients) == sorted(services_to_demo), sorted(clients)
    print(f"\nSuccessfully created {len(clients)} AWS service clients")
    
    # Show regional capabilities
    print(f"\n=== Regional Support ===")
    sample_regions = session.get_available_regions('ec2')
    print(f"EC2 available in {len(sample_regions)} regions")
    print(f"Sample regions: {sample_regions[:8]}")
    
    # Demonstrate credential chain awareness
    print(f"\n=== Credential Chain ===")
    print("Boto3 credential search order:")
    print("1. Explicitly passed credentials")
    print("2. Environment variables (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY)")
    print("3. AWS credentials file (~/.aws/credentials)")
    print("4. AWS config file (~/.aws/config)")
    print("5. IAM role (when running on AWS infrastructure)")
    print("6. Other credential providers")
    
    print(f"\n=== Key Boto3 Features ===")
    print("✓ Low-level client interface - Direct API mapping")
    print("✓ High-level resource interface - Object-oriented abstraction")
    print("✓ Automatic retry logic with exponential backoff")
    print("✓ Pagination for large result sets")
    print("✓ Waiters for resource state polling")
    print("✓ Session management for credential/region isolation")
    print("✓ Comprehensive error handling and exception types")
    print("✓ Thread-safe clients for concurrent usage")
    print("✓ Extensive configuration options")
    print("✓ Support for all AWS regions and services")
    
    print(f"\n=== Common Use Cases ===")
    print("• Infrastructure automation and provisioning")
    print("• Data pipeline and ETL workflows")
    print("• Serverless application development")
    print("• Cloud-native microservices")
    print("• Backup and disaster recovery solutions")
    print("• Machine learning model deployment")
    print("• Real-time data processing")
    print("• DevOps and CI/CD automation")
    
    print("\nBoto3 comprehensive capabilities overview completed successfully!")
    
except Exception as e:
    print(f"Error in comprehensive overview: {e}")
    raise AssertionError(
        f"boto3 example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e